# 31 — Package-Mode Survey Context Fix: Validation Replay

**Question.** Does the `context_policy_id=PACKAGE_SCOPE` patch on `administer_survey()` actually change downstream survey numbers, not just `assemble_context()` byte length?

**Pre-fix bug (confirmed in NB 30 + transcript audit).** In package mode, `manage_memory` writes reflections and `daily_summaries` keyed under `PACKAGE_SCOPE`, but `administer_survey` was calling `assemble_context(policy_id=<single policy>)`. Since `assemble_context` filters by exact `policy_id` match, package-scoped entries were dropped → per-policy survey context was bit-identical across days for every agent and every policy.

**Fix (Option B).** `administer_survey(... context_policy_id=None)` and `run_end_of_day_survey(... context_policy_id=None)` thread a separate `ctx_policy` into the system prompt; `sim.py` package-mode branch passes `PACKAGE_SCOPE`. Storage keys still use the per-policy `policy_id`.

**Approach.**
- Reuse the same 20 stratified agents from NB 30 (10 A-only + 10 B-only), R14 split50 source, frozen state hydrated via `inject_qwen_state`.
- Run a **post-fix** replay (`replay_survey_pkg`) that builds the system prompt with `policy_id=PACKAGE_SCOPE` — same behavior the patched production path now uses.
- Two models, chosen to isolate the fix's effect from model-character noise:
  - **Qwen3-8B (local)** — same model R14 used; pre-fix baseline lives in R14's `opinion_trajectories.csv`.
  - **gpt-5.4-mini** — in NB 30 this was the closest to Qwen (mean |Δ|=0.27), so post-fix divergence is more attributable to the memory injection than to a different model voice.

**Comparison axes.**
| condition | source |
|---|---|
| Qwen pre-fix | `data/output/experiments/run_6202348_R14_split50_5day/20260619_192129/opinion_trajectories.csv` |
| GPT pre-fix | `data/output/calibration/30_replay_20260620_162106/results.csv` (NB 30) |
| Qwen post-fix | this notebook |
| GPT post-fix | this notebook |

**Budget.** 2 models × 20 agents × 3 days × 6 policies × 2 debias steps = **1,440 calls** (~720 local Qwen + 720 paid GPT, ~$2–4).

**Diagnostic guarantee (Cell 6).** Asserts that `md5(system_prompt @ day 1) != md5(system_prompt @ day 2) != md5(system_prompt @ day 3)` for the PACKAGE_SCOPE context on a sample agent — fails loudly if the fix's effect doesn't reach the prompt. Also explicitly shows the *old* per-policy context md5s are bit-identical across days (= the bug being fixed).

**No agent state is mutated** during the replay. The cumulative results CSV is written atomically and the matrix loop is idempotent.

---

## How to use
1. **Start the local Qwen server** (vLLM/llama.cpp at `http://localhost:8000/v1` serving `Qwen/Qwen3-8B`) before running Cell 9 or beyond.
2. **Cells 1–7** are setup; safe to run unconditionally.
3. **Cell 8** is a 1-call smoke per model — confirms wiring & that the local server responds.
4. **Cell 9** sets `DRY_RUN = True` by default. Flip to `False` for the full 1440-call run.
5. **Cell 10** is the matrix execution. Idempotent on `(model_label, agent_id, day, policy_id)`.
6. **Cells 11–13** are analysis (4-condition trajectory, per-cell divergence, prose comparison).

In [1]:
# Cell 1 — Imports + paths
from __future__ import annotations

import hashlib
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd()
REPO = _here if (_here / "src" / "cag").exists() else _here.parent
assert (REPO / "src" / "cag").exists(), f"Could not find src/cag from {_here}"
sys.path.insert(0, str(REPO / "src"))

from cag.abm.agent import (
    SurveyedCitizen,
    _ANTI_SYCOPHANCY,
    _DEBIAS_STEP1_TEMPLATE,
    _DEBIAS_STEP2_TEMPLATE,
)
from cag.abm.attributes.opinion import (
    ALL_CLIMATE_POLICIES,
    ClimatePolicyID,
    PACKAGE_SCOPE,
    RESPONSE_LABELS,
    RESPONSE_SCALE,
    SURVEY_QUESTIONS,
)
from cag.io.llm import (
    configure_local,
    load_api_key,
    parse_letter_response,
    ping_local,
    send_chat,
)
from cag.io.survey import load

# Frozen source (same as NB 30).
SRC_RUN = REPO / "data" / "output" / "experiments" / "run_6202348_R14_split50_5day" / "20260619_192129"
assert SRC_RUN.exists(), f"Missing R14 split50 source dir: {SRC_RUN}"

# NB 30 results (used as the GPT pre-fix baseline). Override if you want a different NB 30 stamp.
NB30_RESULTS = REPO / "data" / "output" / "calibration" / "30_replay_20260620_162106" / "results.csv"

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = REPO / "data" / "output" / "calibration" / f"31_pkgfix_{STAMP}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPO          : {REPO}")
print(f"SRC_RUN       : {SRC_RUN}")
print(f"NB30_RESULTS  : {NB30_RESULTS}  exists={NB30_RESULTS.exists()}")
print(f"OUT_DIR       : {OUT_DIR}")

REPO          : /Users/vbwt265/src/GitHub/Climate-Action-GABM
SRC_RUN       : /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/experiments/run_6202348_R14_split50_5day/20260619_192129
NB30_RESULTS  : /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/30_replay_20260620_162106/results.csv  exists=True
OUT_DIR       : /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/31_pkgfix_20260620_193325


In [2]:
# Cell 2 — Load Qwen artefacts from R14 split50.
REF_CSV  = SRC_RUN / "reflections.csv"
SUM_CSV  = SRC_RUN / "daily_summaries.csv"
SR_CSV   = SRC_RUN / "survey_reasoning.csv"
OT_CSV   = SRC_RUN / "opinion_trajectories.csv"
MSG_CSV  = SRC_RUN / "messages.csv"
GT_CSV   = SRC_RUN / "ground_truth.csv"
CFG_JSON = SRC_RUN / "config.json"

qwen_ref = pd.read_csv(REF_CSV)
qwen_sum = pd.read_csv(SUM_CSV)
qwen_sr  = pd.read_csv(SR_CSV)
qwen_ot  = pd.read_csv(OT_CSV)
qwen_msg = pd.read_csv(MSG_CSV)
qwen_gt  = pd.read_csv(GT_CSV)
qwen_cfg = json.loads(CFG_JSON.read_text())

print(f"reflections        : {len(qwen_ref):>5}  agents={qwen_ref['agent_id'].nunique()}  days={sorted(qwen_ref['day'].unique().tolist())}")
print(f"daily_summaries    : {len(qwen_sum):>5}  agents={qwen_sum['agent_id'].nunique()}  days={sorted(qwen_sum['day'].unique().tolist())}")
print(f"survey_reasoning   : {len(qwen_sr):>5}  agents={qwen_sr['agent_id'].nunique()}  days={sorted(qwen_sr['day'].unique().tolist())}")
print(f"opinion_trajectory : {len(qwen_ot):>5}  agents={qwen_ot['agent_id'].nunique()}  days={sorted(qwen_ot['day'].unique().tolist())}")
print(f"messages           : {len(qwen_msg):>5}")
print(f"ground_truth       : {len(qwen_gt):>5}")
print(f"config provider    : {qwen_cfg.get('llm_provider')!r}, model {qwen_cfg.get('llm_model')!r}")
print(f"comm_mode          : {qwen_cfg.get('communication_mode')!r}")

reflections        :   250  agents=50  days=[1, 2, 3, 4, 5]
daily_summaries    :   150  agents=50  days=[1, 2, 3]
survey_reasoning   :  1800  agents=50  days=[0, 1, 2, 3, 4, 5]
opinion_trajectory :  1800  agents=50  days=[0, 1, 2, 3, 4, 5]
messages           :   250
ground_truth       :   300
config provider    : 'local', model 'Qwen/Qwen3-8B'
comm_mode          : 'package'


In [3]:
# Cell 3 — Rebuild nation, infer buckets, stratified sample of 20 agents.
# Uses the same RNG seeds as NB 30 so the agent set is identical.

from cag.__main__ import build_nation

RANDOM_SEED_NATION = qwen_cfg.get("random_seed", 42)
RANDOM_SEED_SUBSET = 123
N_PER_BUCKET       = 10

YOUGOV_CSV = REPO / "data" / "yougov_survey_data" / "YouGovProcessedData.csv"
assert YOUGOV_CSV.exists(), f"Missing YouGov CSV: {YOUGOV_CSV}"

data_full = load(str(YOUGOV_CSV))
data50 = data_full.sample(n=qwen_cfg["n_citizens"], random_state=RANDOM_SEED_NATION).reset_index(drop=True)
nation50 = build_nation(data50)
print(f"Rebuilt nation with {len(nation50.agents_active)} agents.")

broadcast_ref = qwen_ref[qwen_ref["phase"].isin(["P-A", "P-B"])]
phase_per_agent = broadcast_ref.groupby("agent_id")["phase"].first()
bucket_map = phase_per_agent.map({"P-A": "A-only", "P-B": "B-only"})
print("\nBucket distribution (full 50):")
print(bucket_map.value_counts())

rng = np.random.default_rng(RANDOM_SEED_SUBSET)
a_pool = bucket_map[bucket_map == "A-only"].index.tolist()
b_pool = bucket_map[bucket_map == "B-only"].index.tolist()
a_pick = list(rng.choice(a_pool, size=N_PER_BUCKET, replace=False))
b_pick = list(rng.choice(b_pool, size=N_PER_BUCKET, replace=False))
PICKED = sorted(a_pick + b_pick)
PICKED_BUCKET = {aid: ("A-only" if aid in a_pick else "B-only") for aid in PICKED}
print(f"\nPicked {len(PICKED)} agents ({len(a_pick)} A-only + {len(b_pick)} B-only).")
print(f"Agent IDs (first 5): {PICKED[:5]}...")

Rebuilt nation with 50 agents.

Bucket distribution (full 50):
phase
B-only    25
A-only    25
Name: count, dtype: int64

Picked 20 agents (10 A-only + 10 B-only).
Agent IDs (first 5): [np.float64(69.0), np.float64(91.0), np.float64(165.0), np.float64(166.0), np.float64(238.0)]...


In [4]:
# Cell 4 — inject_qwen_state(): hydrate the SurveyedCitizen with Qwen's state.
# Same helper as NB 30 Cell 4.

_PID_FROM_STR = {str(pid): pid for pid in ALL_CLIMATE_POLICIES}
_PID_FROM_STR[PACKAGE_SCOPE] = PACKAGE_SCOPE

def _to_policy(value):
    if value is None or value == "" or (isinstance(value, float) and pd.isna(value)):
        return ""
    return _PID_FROM_STR.get(str(value), str(value))

def inject_qwen_state(agent, agent_id, ref_df, sum_df, sr_df, ot_df):
    aid = agent_id
    agent.reflections = []
    for row in ref_df[ref_df["agent_id"] == aid].itertuples(index=False):
        try:
            messages_received = json.loads(row.messages_received_json)
        except (TypeError, ValueError):
            messages_received = []
        try:
            policy_ids = json.loads(row.policy_ids_json)
        except (TypeError, ValueError):
            policy_ids = []
        entry = {
            "day": int(row.day),
            "phase": row.phase,
            "policy_id": _to_policy(row.policy_id),
            "text": row.text,
            "messages_received": messages_received,
        }
        if policy_ids:
            entry["policy_ids"] = [_to_policy(p) for p in policy_ids]
        agent.reflections.append(entry)
    agent.daily_summaries = {}
    for row in sum_df[sum_df["agent_id"] == aid].itertuples(index=False):
        agent.daily_summaries[(int(row.day), _to_policy(row.policy_id))] = row.summary
    agent.survey_reasoning = {}
    for row in sr_df[sr_df["agent_id"] == aid].itertuples(index=False):
        pid = _to_policy(row.policy_id)
        agent.survey_reasoning.setdefault(pid, []).append((int(row.day), row.reasoning))
    agent.opinion_history = {}
    for row in ot_df[ot_df["agent_id"] == aid].itertuples(index=False):
        pid = _to_policy(row.policy_id)
        agent.opinion_history.setdefault(pid, []).append((int(row.day), int(row.numeric)))
    return agent

for aid in PICKED:
    inject_qwen_state(
        nation50.agents_active[aid], aid,
        qwen_ref, qwen_sum, qwen_sr, qwen_ot,
    )
print(f"Hydrated {len(PICKED)} agents from Qwen state.")

sample = nation50.agents_active[PICKED[0]]
print(f"\nSample agent {PICKED[0]} ({PICKED_BUCKET[PICKED[0]]}):")
print(f"  reflections        : {len(sample.reflections):>3}  (days {sorted({r['day'] for r in sample.reflections})})")
print(f"  daily_summaries    : {len(sample.daily_summaries):>3}  keys {sorted(sample.daily_summaries.keys())}")
print(f"  survey_reasoning   : {len(sample.survey_reasoning):>3} policies")
print(f"  opinion_history    : {len(sample.opinion_history):>3} policies")

Hydrated 20 agents from Qwen state.

Sample agent 69.0 (B-only):
  reflections        :   5  (days [1, 2, 3, 4, 5])
  daily_summaries    :   3  keys [(1, 'climate_policy_package'), (2, 'climate_policy_package'), (3, 'climate_policy_package')]
  survey_reasoning   :   6 policies
  opinion_history    :   6 policies


In [5]:
# Cell 5 — Same substring sanity as NB 30 Cell 5. Confirms hydration worked.

def _first_words(text, n=20):
    return " ".join(str(text).split()[:n])

PROBE_POLICY = ClimatePolicyID.RENEWABLE_ENERGY
checked, ok = 0, 0
for aid in PICKED[:5]:
    agent = nation50.agents_active[aid]
    ctx1 = agent.assemble_context(day=1, policy_id=PROBE_POLICY)
    ctx3 = agent.assemble_context(day=3, policy_id=PROBE_POLICY)
    ctx3_pkg = agent.assemble_context(day=3, policy_id=PACKAGE_SCOPE)
    day0_text = None
    for d, t in agent.survey_reasoning.get(PROBE_POLICY, []):
        if d == 0:
            day0_text = t
            break
    assert day0_text is not None, f"agent {aid} has no Day-0 rationale for {PROBE_POLICY}"
    snippet = _first_words(day0_text, 8)
    in1 = snippet in ctx1
    in3 = snippet in ctx3
    day2_ref = [r for r in agent.reflections if r["day"] == 2]
    ref_snippet = _first_words(day2_ref[0]["text"], 8) if day2_ref else ""
    ref_in3_pkg = ref_snippet in ctx3_pkg if ref_snippet else True
    checked += 1
    if in1 and in3 and ref_in3_pkg:
        ok += 1
    else:
        print(f"  agent {aid}: day0_in_ctx1={in1}, day0_in_ctx3={in3}, day2ref_in_ctx3_pkg={ref_in3_pkg}")
print(f"\nSubstring sanity: {ok}/{checked} agents passed all three checks.")
assert ok == checked, "State injection sanity check failed; aborting before LLM spend."


Substring sanity: 5/5 agents passed all three checks.


In [6]:
# Cell 6 — Proof-of-fix diagnostic.
#
# Hash the system prompt the patched survey path now sees (policy_id=PACKAGE_SCOPE)
# across days 1, 2, 3 — must DIFFER day-to-day, since each day's manage_memory
# adds reflections / a daily summary. Also hash the OLD per-policy context
# (policy_id=<policy>) — these are expected to be bit-identical across days,
# which is the bug we are fixing.

def _md5(s):
    return hashlib.md5((s or "").encode("utf-8")).hexdigest()[:10]

diag_rows = []
for aid in PICKED[:5]:
    agent = nation50.agents_active[aid]
    for policy in [ClimatePolicyID.RENEWABLE_ENERGY, ClimatePolicyID.CARBON_TAX]:
        for day in [1, 2, 3]:
            ctx_old = agent.get_system_prompt(day=day, policy_id=policy)        # pre-fix path
            ctx_new = agent.get_system_prompt(day=day, policy_id=PACKAGE_SCOPE) # post-fix path
            diag_rows.append({
                "agent_id": aid, "policy": str(policy), "day": day,
                "len_pre": len(ctx_old),  "md5_pre": _md5(ctx_old),
                "len_post": len(ctx_new), "md5_post": _md5(ctx_new),
            })
diag = pd.DataFrame(diag_rows)
print(diag.to_string(index=False))

# Per-(agent, policy) check: pre-fix md5s must all match across days;
# post-fix md5s must all differ across days.
fail = []
for (aid, pol), sub in diag.groupby(["agent_id", "policy"]):
    n_pre  = sub["md5_pre"].nunique()
    n_post = sub["md5_post"].nunique()
    if n_pre != 1 or n_post != len(sub):
        fail.append((aid, pol, n_pre, n_post))
if fail:
    print("\nFAIL — diagnostic broke:")
    for f in fail:
        print(f"  agent={f[0]} policy={f[1]} n_unique_pre={f[2]} n_unique_post={f[3]}")
    raise AssertionError("Pre-fix should be bit-identical across days; post-fix must vary across days.")
print("\nOK — pre-fix per-policy contexts are bit-identical across days (the bug)")
print("     post-fix PACKAGE_SCOPE contexts vary across days (the fix)")

 agent_id             policy  day  len_pre    md5_pre  len_post   md5_post
     69.0 ClimatePolicyID(1)    1     2695 ba6e41fd01      7588 fccc47f33b
     69.0 ClimatePolicyID(1)    2     2695 ba6e41fd01      9577 d02ca604ef
     69.0 ClimatePolicyID(1)    3     2695 ba6e41fd01     10181 b703146f78
     69.0 ClimatePolicyID(5)    1     2750 7ca5005e3c      7588 fccc47f33b
     69.0 ClimatePolicyID(5)    2     2750 7ca5005e3c      9577 d02ca604ef
     69.0 ClimatePolicyID(5)    3     2750 7ca5005e3c     10181 b703146f78
     91.0 ClimatePolicyID(1)    1     2672 7e849f65f7      7169 a91312313b
     91.0 ClimatePolicyID(1)    2     2672 7e849f65f7      9171 f68c15dde0
     91.0 ClimatePolicyID(1)    3     2672 7e849f65f7      9415 fd9e874b4b
     91.0 ClimatePolicyID(5)    1     2622 01712a80c4      7169 a91312313b
     91.0 ClimatePolicyID(5)    2     2622 01712a80c4      9171 f68c15dde0
     91.0 ClimatePolicyID(5)    3     2622 01712a80c4      9415 fd9e874b4b
    165.0 ClimatePolicyID

In [8]:
# Cell 7 — replay_survey_pkg(): the 2-step debias chain with the POST-FIX
# context wiring. Mirrors administer_survey() with context_policy_id=PACKAGE_SCOPE
# but returns instead of writing — no agent state is mutated.
#
# This is byte-for-byte the same prompt construction the patched production
# path now uses in sim.py's package-mode end-of-day survey loop.

def replay_survey_pkg(agent, day, policy_id, model, provider, api_key=None,
                      temperature=0.5, thinking=False):
    out = {
        "reasoning": None, "raw_response": None, "letter": None,
        "numeric": None, "latency_s": None, "error": None,
    }
    t0 = time.perf_counter()
    try:
        # POST-FIX: system prompt sees PACKAGE_SCOPE memory; question is per-policy.
        system_prompt = agent.get_system_prompt(day=day, policy_id=PACKAGE_SCOPE)
        policy_question = SURVEY_QUESTIONS[policy_id]
        step1_prompt = _DEBIAS_STEP1_TEMPLATE.format(
            anti_sycophancy=_ANTI_SYCOPHANCY,
            policy_question=policy_question,
        )
        reasoning = send_chat(
            system_prompt, step1_prompt, api_key=api_key, model=model,
            provider=provider, temperature=temperature, thinking=thinking,
        )
        response_options = "\n".join(
            f"{letter}. {label}" for letter, label in RESPONSE_LABELS.items()
        )
        step2_system = system_prompt + "\n\nYour reasoning about this policy:\n" + reasoning
        step2_prompt = _DEBIAS_STEP2_TEMPLATE.format(
            policy_question=policy_question,
            response_options=response_options,
        )
        raw = send_chat(
            step2_system, step2_prompt, api_key=api_key, model=model,
            provider=provider, temperature=temperature, thinking=thinking,
        )
        letter = parse_letter_response(raw)
        numeric = RESPONSE_SCALE.get(letter)
        out.update(reasoning=reasoning, raw_response=raw, letter=letter, numeric=numeric)
    except Exception as exc:
        out["error"] = f"{type(exc).__name__}: {exc}"
    finally:
        out["latency_s"] = time.perf_counter() - t0
    return out


In [9]:
# Cell 8 — Single-call smoke against the local Qwen server (and GPT if key present).
# Confirms the Qwen server is live and the GPT key works before the full matrix.
#
# NOTE: override LOCAL_BASE_URL / LOCAL_MODEL to match your server.
# - mlx_lm.server default port is 8080; vLLM in R14 used 8000.
# - The model string MUST match `--model` exactly (e.g. mlx-community/Qwen3-8B-4bit
#   for the 4-bit MLX quant, or Qwen/Qwen3-8B for FP).
# If you run a different quant than R14 (FP), the Qwen pre/post divergence in
# Cell 12 will include some quantization noise on top of the fix's effect; the
# GPT pre/post divergence remains a clean fix-only signal.

LOCAL_BASE_URL = "http://localhost:8080/v1"          # mlx_lm.server default
LOCAL_MODEL    = "mlx-community/Qwen3-8B-4bit"       # match `--model` flag exactly

configure_local(base_url=LOCAL_BASE_URL, extra_body=qwen_cfg.get("local_extra_body"))
try:
    ping_local()
    print(f"OK  local Qwen server responding at {LOCAL_BASE_URL}")
except Exception as exc:
    print(f"FAIL  local Qwen server not reachable at {LOCAL_BASE_URL}: {exc}")
    print("      Start the server before running Cells 9–10.")

SMOKE_AGENT  = PICKED[0]
SMOKE_DAY    = 1
SMOKE_POLICY = ClimatePolicyID.RENEWABLE_ENERGY

for prov, mdl in [("local", LOCAL_MODEL), ("openai", "gpt-5.4-mini")]:
    try:
        key = load_api_key(prov)
    except Exception as exc:
        print(f"\n{prov}: no key ({exc}) — skipping smoke")
        continue
    if not key and prov != "local":
        print(f"\n{prov}: empty key — skipping smoke")
        continue
    print(f"\nSmoke: provider={prov!r} model={mdl!r} agent={SMOKE_AGENT} day={SMOKE_DAY} policy={SMOKE_POLICY!r}")
    res = replay_survey_pkg(
        nation50.agents_active[SMOKE_AGENT], SMOKE_DAY, SMOKE_POLICY,
        model=mdl, provider=prov, api_key=key,
    )
    if res["error"]:
        print(f"  ERROR: {res['error']}")
    else:
        print(f"  letter={res['letter']!r}  numeric={res['numeric']}  latency={res['latency_s']:.2f}s")
        print(f"  reasoning (first 240 chars): {(res['reasoning'] or '')[:240]}")


OK  local Qwen server responding at http://localhost:8080/v1

Smoke: provider='local' model='mlx-community/Qwen3-8B-4bit' agent=69.0 day=1 policy=ClimatePolicyID(1)
  letter='E'  numeric=1  latency=31.98s
  reasoning (first 240 chars): I would slightly support accelerating the roll-out of renewable energy production because I believe in protecting the environment and ensuring a sustainable future for all. However, I am cautious about the pace and cost of such projects, es

Smoke: provider='openai' model='gpt-5.4-mini' agent=69.0 day=1 policy=ClimatePolicyID(1)


  letter='E'  numeric=1  latency=3.65s
  reasoning (first 240 chars): I would **slightly support** accelerating renewable energy roll-out. I care a lot about protecting the environment and supporting a sustainable future, and as a Remain-voting, centre-leaning Conservative I’m open to practical climate action


In [11]:
# Cell 9 — Model registry + run-mode switches.
#
# DRY_RUN=True  → 1 agent × 1 day × 1 policy × 2 models × 2 steps = 4 LLM calls
# DRY_RUN=False → full 1440-call matrix (720 local Qwen + 720 paid GPT)
#
# The qwen3-8b label in Cell 11/12 joins to R14 saved opinion_trajectories
# regardless of the local model string used, so a 4-bit MLX quant is fine —
# but its pre/post divergence will include quantization noise (see Cell 8 note).
# The gpt-5.4-mini row gives the clean fix-only signal.

DRY_RUN = False   # ← flip to False for the full run

MODEL_REGISTRY = [
    {"label": "qwen3-8b",     "provider": "local",  "model": LOCAL_MODEL},
    {"label": "gpt-5.4-mini", "provider": "openai", "model": "gpt-5.4-mini"},
]

DAYS     = [1, 2, 3]
POLICIES = list(ALL_CLIMATE_POLICIES)

ACTIVE_MODELS = []
for entry in MODEL_REGISTRY:
    try:
        key = load_api_key(entry["provider"])
    except Exception as exc:
        print(f"  skip {entry['label']!r}: no key ({exc})")
        continue
    if entry["provider"] != "local" and not key:
        print(f"  skip {entry['label']!r}: empty key for provider {entry['provider']!r}")
        continue
    ACTIVE_MODELS.append({**entry, "api_key": key})

if DRY_RUN:
    AGENTS_RUN   = PICKED[:1]
    DAYS_RUN     = DAYS[:1]
    POLICIES_RUN = POLICIES[:1]
else:
    AGENTS_RUN   = PICKED
    DAYS_RUN     = DAYS
    POLICIES_RUN = POLICIES

expected_calls = len(ACTIVE_MODELS) * len(AGENTS_RUN) * len(DAYS_RUN) * len(POLICIES_RUN) * 2
print(f"\nDRY_RUN={DRY_RUN}")
print(f"Active models : {[(m['label'], m['model']) for m in ACTIVE_MODELS]}")
print(f"Agents        : {len(AGENTS_RUN)}")
print(f"Days          : {DAYS_RUN}")
print(f"Policies      : {len(POLICIES_RUN)}")
print(f"Expected LLM calls (2 per cell): {expected_calls}")



DRY_RUN=False
Active models : [('qwen3-8b', 'mlx-community/Qwen3-8B-4bit'), ('gpt-5.4-mini', 'gpt-5.4-mini')]
Agents        : 20
Days          : [1, 2, 3]
Policies      : 6
Expected LLM calls (2 per cell): 1440


In [12]:
# Cell 10 — Matrix execution (post-fix replay). Idempotent on
# (model_label, agent_id, day, policy_id). Atomic write after each model.

RESULTS_CSV = OUT_DIR / "results.csv"
RESULTS_COLS = [
    "model_label", "provider", "model", "agent_id", "bucket",
    "day", "policy_id", "reasoning", "raw_response",
    "letter", "numeric", "latency_s", "error",
]

if RESULTS_CSV.exists():
    existing = pd.read_csv(RESULTS_CSV)
    done_keys = set(zip(existing["model_label"], existing["agent_id"], existing["day"], existing["policy_id"]))
    rows = existing.to_dict("records")
    print(f"Resuming: {len(rows)} rows already on disk; will skip duplicates.")
else:
    done_keys = set()
    rows = []

def _atomic_dump():
    df = pd.DataFrame(rows, columns=RESULTS_COLS)
    tmp = RESULTS_CSV.with_suffix(".csv.tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(RESULTS_CSV)

overall_t0 = time.perf_counter()
for m in ACTIVE_MODELS:
    label, prov, mdl, key = m["label"], m["provider"], m["model"], m["api_key"]
    print(f"\n=== {label}  ({prov} / {mdl}) ===")
    model_t0 = time.perf_counter()
    n_done, n_skip, n_err = 0, 0, 0
    for aid in AGENTS_RUN:
        agent = nation50.agents_active[aid]
        bucket = PICKED_BUCKET[aid]
        for day in DAYS_RUN:
            for policy in POLICIES_RUN:
                key_tuple = (label, aid, day, str(policy))
                if key_tuple in done_keys:
                    n_skip += 1
                    continue
                res = replay_survey_pkg(agent, day, policy, model=mdl, provider=prov, api_key=key)
                rows.append({
                    "model_label": label, "provider": prov, "model": mdl,
                    "agent_id": aid, "bucket": bucket,
                    "day": day, "policy_id": str(policy),
                    "reasoning": res["reasoning"], "raw_response": res["raw_response"],
                    "letter": res["letter"], "numeric": res["numeric"],
                    "latency_s": res["latency_s"], "error": res["error"],
                })
                done_keys.add(key_tuple)
                n_done += 1
                if res["error"]:
                    n_err += 1
    _atomic_dump()
    elapsed = time.perf_counter() - model_t0
    print(f"  {label}: {n_done} done, {n_skip} skipped, {n_err} errors  [{elapsed:.1f}s]")

total_elapsed = time.perf_counter() - overall_t0
print(f"\nAll models complete. {len(rows)} total rows, {total_elapsed:.1f}s. Wrote {RESULTS_CSV}.")


=== qwen3-8b  (local / mlx-community/Qwen3-8B-4bit) ===
  qwen3-8b: 360 done, 0 skipped, 0 errors  [6274.2s]

=== gpt-5.4-mini  (openai / gpt-5.4-mini) ===
  gpt-5.4-mini: 360 done, 0 skipped, 0 errors  [778.0s]

All models complete. 720 total rows, 7052.3s. Wrote /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/31_pkgfix_20260620_193325/results.csv.


In [13]:
# Cell 11 — 4-condition bucket trajectory plot.
# Overlays for each bucket:
#   - Qwen pre-fix  (R14 saved opinion_trajectories.csv, subset to PICKED agents)
#   - GPT  pre-fix  (NB 30 results.csv if present)
#   - Qwen post-fix (this notebook)
#   - GPT  post-fix (this notebook)

import matplotlib.pyplot as plt

res_df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(columns=RESULTS_COLS)
if res_df.empty:
    print("No post-fix replay results on disk yet — run Cell 10.")
else:
    post_clean = res_df.dropna(subset=["numeric"]).copy()
    post_clean["condition"] = post_clean["model_label"] + " post-fix"

    # Qwen pre-fix (from R14 opinion_trajectories.csv).
    qw_pre = qwen_ot[qwen_ot["agent_id"].isin(PICKED) & qwen_ot["day"].isin([0] + list(DAYS))].copy()
    qw_pre["bucket"] = qw_pre["agent_id"].map(PICKED_BUCKET)
    qw_pre["condition"] = "qwen3-8b pre-fix (R14 saved)"
    qw_pre = qw_pre.rename(columns={"numeric": "numeric"})[["agent_id", "day", "bucket", "numeric", "condition"]]

    # GPT pre-fix (from NB 30 results.csv, if present).
    if NB30_RESULTS.exists():
        nb30 = pd.read_csv(NB30_RESULTS)
        gpt_pre = nb30[(nb30["model_label"] == "gpt-5.4-mini") & nb30["agent_id"].isin(PICKED)].copy()
        gpt_pre = gpt_pre.dropna(subset=["numeric"])
        gpt_pre["condition"] = "gpt-5.4-mini pre-fix (NB 30)"
        gpt_pre = gpt_pre[["agent_id", "day", "bucket", "numeric", "condition"]]
    else:
        gpt_pre = pd.DataFrame(columns=["agent_id", "day", "bucket", "numeric", "condition"])
        print("  (NB 30 results not found; GPT pre-fix line will be omitted)")

    panel = pd.concat([
        qw_pre,
        gpt_pre,
        post_clean[["agent_id", "day", "bucket", "numeric", "condition"]],
    ], ignore_index=True)
    means = panel.groupby(["condition", "day", "bucket"])["numeric"].mean().reset_index()

    style_map = {
        "qwen3-8b pre-fix (R14 saved)":  ("x", ":",  "black"),
        "gpt-5.4-mini pre-fix (NB 30)":  ("s", ":",  "grey"),
        "qwen3-8b post-fix":             ("o", "-",  "tab:blue"),
        "gpt-5.4-mini post-fix":         ("o", "-",  "tab:orange"),
    }
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    for ax, bucket in zip(axes, ["A-only", "B-only"]):
        for cond, sub in means[means["bucket"] == bucket].groupby("condition"):
            sub = sub.sort_values("day")
            marker, linestyle, color = style_map.get(cond, ("o", "-", None))
            ax.plot(sub["day"], sub["numeric"], marker=marker, linestyle=linestyle,
                    color=color, label=cond)
        ax.set_title(bucket)
        ax.set_xlabel("Day")
        ax.axhline(0, color="grey", linewidth=0.5)
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Mean numeric opinion (–3..+3)")
    axes[1].legend(loc="best", fontsize=8)
    fig.suptitle(f"NB 31 — Pre-fix vs post-fix bucket-mean opinion (n={len(PICKED)//2} per bucket, 6 policies pooled)")
    fig.tight_layout()
    plt.savefig(OUT_DIR / "bucket_trajectory_4condition.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved {OUT_DIR / 'bucket_trajectory_4condition.png'}")

Saved /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/31_pkgfix_20260620_193325/bucket_trajectory_4condition.png


/var/folders/9r/wv909y5d5tz8vdmlx6gll_2h0000gr/T/ipykernel_83125/2230933389.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Cell 12 — Per-cell divergence: post-fix vs pre-fix for each model.
# Headline metric: for each model, |post_fix_numeric − pre_fix_numeric| over
# the cells that exist in BOTH conditions. If this is mostly 0, the fix had
# no observable effect for that model; if it's >> 0, the fix changed answers.

if res_df.empty:
    print("No post-fix replay results on disk yet — run Cell 10.")
else:
    post = res_df.dropna(subset=["numeric"]).copy()
    post["policy_id"] = post["policy_id"].astype(str)

    pre_qwen = qwen_ot[qwen_ot["agent_id"].isin(PICKED) & qwen_ot["day"].isin(DAYS)].copy()
    pre_qwen["policy_id"] = pre_qwen["policy_id"].astype(str)
    pre_qwen = pre_qwen.rename(columns={"numeric": "pre_numeric"})[["agent_id", "day", "policy_id", "pre_numeric"]]
    pre_qwen["model_label"] = "qwen3-8b"

    if NB30_RESULTS.exists():
        nb30 = pd.read_csv(NB30_RESULTS)
        pre_gpt = nb30[(nb30["model_label"] == "gpt-5.4-mini") & nb30["agent_id"].isin(PICKED)].copy()
        pre_gpt = pre_gpt.dropna(subset=["numeric"]).rename(columns={"numeric": "pre_numeric"})
        pre_gpt["policy_id"] = pre_gpt["policy_id"].astype(str)
        pre_gpt = pre_gpt[["agent_id", "day", "policy_id", "pre_numeric"]]
        pre_gpt["model_label"] = "gpt-5.4-mini"
    else:
        pre_gpt = pd.DataFrame(columns=["agent_id", "day", "policy_id", "pre_numeric", "model_label"])

    pre_all = pd.concat([pre_qwen, pre_gpt], ignore_index=True)
    joined = post.merge(pre_all, on=["model_label", "agent_id", "day", "policy_id"], how="inner")
    joined["abs_diff"] = (joined["numeric"] - joined["pre_numeric"]).abs()
    joined["signed_diff"] = joined["numeric"] - joined["pre_numeric"]

    summary = (joined.groupby("model_label")
                     .agg(n=("abs_diff", "size"),
                          changed_share=("abs_diff", lambda s: (s > 0).mean()),
                          median_abs=("abs_diff", "median"),
                          mean_abs=("abs_diff", "mean"),
                          p90_abs=("abs_diff", lambda s: s.quantile(0.9)),
                          mean_signed=("signed_diff", "mean"))
                     .round(3))
    print("Per-cell |post-fix − pre-fix| numeric, by model:")
    print(summary.to_string())
    summary.to_csv(OUT_DIR / "prepost_divergence_summary.csv")
    print(f"\nSaved {OUT_DIR / 'prepost_divergence_summary.csv'}")

    # Day-by-day mean signed shift (positive = post-fix is more pro-climate).
    by_day = (joined.groupby(["model_label", "day"])["signed_diff"].mean().round(3).unstack("day"))
    print("\nMean signed shift (post − pre) by day:")
    print(by_day.to_string())

Per-cell |post-fix − pre-fix| numeric, by model:
                n  changed_share  median_abs  mean_abs  p90_abs  mean_signed
model_label                                                                 
gpt-5.4-mini  360          0.297         0.0     0.433      1.0       -0.194
qwen3-8b      360          0.331         0.0     0.436      1.0       -0.103

Saved /Users/vbwt265/src/GitHub/Climate-Action-GABM/data/output/calibration/31_pkgfix_20260620_193325/prepost_divergence_summary.csv

Mean signed shift (post − pre) by day:
day               1      2      3
model_label                      
gpt-5.4-mini -0.167 -0.283 -0.133
qwen3-8b     -0.117 -0.158 -0.033


In [15]:
# Cell 13 — Prose comparison: 2 agents × 1 day × 1 policy, side-by-side
# rationales across (Qwen pre-fix saved) ↔ (Qwen post-fix) ↔ (GPT post-fix).

PROSE_DAY    = 3
PROSE_POLICY = ClimatePolicyID.CARBON_TAX

if res_df.empty:
    print("No post-fix replay results on disk yet — run Cell 10.")
else:
    pick_a = sorted([a for a in PICKED if PICKED_BUCKET[a] == "A-only"])[:1]
    pick_b = sorted([a for a in PICKED if PICKED_BUCKET[a] == "B-only"])[:1]
    prose_agents = pick_a + pick_b
    for aid in prose_agents:
        bucket = PICKED_BUCKET[aid]
        print("=" * 88)
        print(f"AGENT {aid}  ({bucket})  —  Day {PROSE_DAY}  —  {PROSE_POLICY!r}")
        print("=" * 88)
        # Qwen pre-fix (saved reasoning from R14).
        qwen_row = qwen_sr[(qwen_sr["agent_id"] == aid) & (qwen_sr["day"] == PROSE_DAY) & (qwen_sr["policy_id"] == str(PROSE_POLICY))]
        qwen_num_row = qwen_ot[(qwen_ot["agent_id"] == aid) & (qwen_ot["day"] == PROSE_DAY) & (qwen_ot["policy_id"] == str(PROSE_POLICY))]
        qwen_num = qwen_num_row["numeric"].iloc[0] if not qwen_num_row.empty else None
        print(f"\n[qwen3-8b pre-fix (R14 saved)] numeric={qwen_num}")
        if not qwen_row.empty:
            print(qwen_row["reasoning"].iloc[0])
        else:
            print("(no Qwen reasoning row found)")
        # Post-fix rows for each model.
        for label in res_df["model_label"].unique():
            sub = res_df[(res_df["model_label"] == label) & (res_df["agent_id"] == aid) & (res_df["day"] == PROSE_DAY) & (res_df["policy_id"] == str(PROSE_POLICY))]
            if sub.empty:
                continue
            row = sub.iloc[0]
            print(f"\n[{label} post-fix] numeric={row['numeric']}  letter={row['letter']!r}")
            print(row["reasoning"] or "(no reasoning)")
        print()

AGENT 91.0  (A-only)  —  Day 3  —  ClimatePolicyID(5)

[qwen3-8b pre-fix (R14 saved)] numeric=2
I would support a carbon tax with revenue distributed to the public because it addresses environmental concerns without disproportionately burdening low-income households, and I believe in fair treatment of all citizens. However, I am cautious about rapid change and prefer stable, traditional approaches, so I would want the policy to be carefully implemented and not disrupt existing industries or ways of life.

[qwen3-8b post-fix] numeric=2  letter='F'
I would **somewhat support** the carbon tax and dividend policy. I see it as a way to address environmental concerns while ensuring fairness, as it distributes the burden across the public and avoids disproportionately impacting lower-income groups. However, I remain cautious about how it might disrupt traditional industries and economic stability, which I value highly.

[gpt-5.4-mini post-fix] numeric=2  letter='F'
I would **somewhat support*